# Fronteiras de Pareto no plano MD × S — cenário aplicado

A distância de Mahalanobis à utopia (MD) é minimizada e a entropia normalizada dos ganhos (S) é maximizada. A dominância é avaliada globalmente entre os quatro métodos: somente pontos que não são dominados por nenhuma solução formam a linha. Para os evolucionários, utiliza-se somente a seed 6 do NSGA-III e a seed 9 do MOEA/D.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks').is_dir() and (p / 'configs').is_dir())
DATA_FILE = REPO / "outputs" / "01a00609-6550-79f2-8328-0714575f0c90" / "solucoes_4_metodos_cenario_aplicado.xlsx"
OUTPUT_DIR = REPO / "results" / "applied" / "figures_dissertation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ["C-NBI", "VRF-NBI", "NSGA-III", "MOEA/D"]
COLORS = {
    "C-NBI": "#E68613",
    "VRF-NBI": "#2878B5",
    "NSGA-III": "#3A923A",
    "MOEA/D": "#D1495B",
}
MARKERS = {"C-NBI": "o", "VRF-NBI": "D", "NSGA-III": "s", "MOEA/D": "^"}
SELECTED_SEEDS = {"NSGA-III": 6, "MOEA/D": 9}
LEGEND_LABELS = {
    "C-NBI": "C-NBI",
    "VRF-NBI": "VRF-NBI",
    "NSGA-III": "NSGA-III (seed 6)",
    "MOEA/D": "MOEA/D (seed 9)",
}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9.5,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
df_all = pd.read_excel(DATA_FILE, sheet_name="Todas as soluções")
mask = (
    df_all["Método"].isin(["C-NBI", "VRF-NBI"])
    | ((df_all["Método"] == "NSGA-III") & (df_all["Semente"] == SELECTED_SEEDS["NSGA-III"]))
    | ((df_all["Método"] == "MOEA/D") & (df_all["Semente"] == SELECTED_SEEDS["MOEA/D"]))
)
df = df_all.loc[mask, ["Método", "Semente", "Solução", "MD", "S"]].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["MD", "S"])

def pareto_md_s(frame, atol=1e-12):
    """Retém pontos não dominados para minimizar MD e maximizar S."""
    ordered = frame.sort_values(["MD", "S"], ascending=[True, False], kind="mergesort")
    keep = []
    best_s = -np.inf
    for idx, row in ordered.iterrows():
        if row["S"] > best_s + atol:
            keep.append(idx)
            best_s = row["S"]
    return ordered.loc[keep].copy()

global_front = pareto_md_s(df).sort_values("MD").copy()
global_counts = global_front["Método"].value_counts()
summary = pd.DataFrame({
    "Método": METHOD_ORDER,
    "Conjunto": ["determinístico", "determinístico", "seed 6", "seed 9"],
    "Soluções": [int((df["Método"] == method).sum()) for method in METHOD_ORDER],
    "Na fronteira global MD×S": [int(global_counts.get(method, 0)) for method in METHOD_ORDER],
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6.30, 4.55), constrained_layout=False)

# Soluções completas em segundo plano.
for method in METHOD_ORDER:
    subset = df.loc[df["Método"] == method]
    ax.scatter(
        subset["MD"], subset["S"],
        s=8, marker=MARKERS[method], color=COLORS[method],
        alpha=0.075, linewidths=0, rasterized=True, zorder=1,
    )

# Uma única fronteira global: todo ponto dominado fica fora da linha.
ax.plot(
    global_front["MD"], global_front["S"],
    color="#555555", linewidth=1.35, alpha=0.9, zorder=3,
)
for method in METHOD_ORDER:
    front = global_front.loc[global_front["Método"] == method]
    ax.scatter(
        front["MD"], front["S"],
        s=25 if method == "C-NBI" else 21,
        marker=MARKERS[method], color=COLORS[method],
        edgecolor="white", linewidth=0.35, zorder=4,
    )

ax.set_xlabel("Distância de Mahalanobis à utopia, MD ↓")
ax.set_ylabel("Entropia normalizada dos ganhos, S ↑")
ax.grid(axis="both", color="#D9D9D9", linewidth=0.55, alpha=0.65)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

x_min, x_max = df["MD"].min(), df["MD"].max()
y_min, y_max = df["S"].min(), df["S"].max()
ax.set_xlim(x_min - 0.02 * (x_max - x_min), x_max + 0.02 * (x_max - x_min))
ax.set_ylim(y_min - 0.035 * (y_max - y_min), y_max + 0.025 * (y_max - y_min))

handles = [
    Line2D([0], [0], color=COLORS[m], marker=MARKERS[m], markersize=5.5,
           linestyle="None", markeredgecolor="white", markeredgewidth=0.35, label=LEGEND_LABELS[m])
    for m in METHOD_ORDER
]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.005), handlelength=1.9, columnspacing=1.5)
fig.subplots_adjust(left=0.13, right=0.985, top=0.98, bottom=0.19)

png_path = OUTPUT_DIR / "fig_pareto_global_md_s_nsga_seed6_moead_seed9.png"
pdf_path = OUTPUT_DIR / "fig_pareto_global_md_s_nsga_seed6_moead_seed9.pdf"
fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.show()

global_front.to_csv(OUTPUT_DIR / "dados_pareto_global_md_s_nsga_seed6_moead_seed9.csv", index=False)
print(png_path)
print(pdf_path)